# CareFlow — AI Care Coordination Assistant

**IIT Hyderabad Applied AI Professional Certification · Capstone**

This notebook is one milestone in the eight-milestone CareFlow implementation.
All patient, insurance, appointment and referral examples are synthetic. No real PHI is used.

> **Data boundary:** the official CareFlow Data Toolkit assets are the source of truth.
> Intake records feed M1; mock API tables feed M2/M6; the policy corpus feeds M3/M4;
> the golden evaluation set is finalized in M8 and should be started at M4.

## M6 — Multi-Agent Coordination + MCP Tool Boundary

### Goal
Separate responsibilities across specialist agents and expose the operational tools through a stable protocol boundary.

The toolkit explicitly maps mock API tables to M2 and M6 and expects the same state to be exposed over MCP.

In [1]:
from dataclasses import dataclass
from typing import Callable, Any

@dataclass
class Tool:
    name: str
    description: str
    handler: Callable[..., Any]

tool_registry = {}

def register_tool(name, description, handler):
    tool_registry[name] = Tool(name, description, handler)

register_tool("get_patient", "Read patient eligibility and plan state", lambda patient_ref: {"patient_ref":patient_ref})
register_tool("get_plan", "Read insurance plan benefits", lambda plan_code: {"plan_code":plan_code})
register_tool("get_appointments", "Read appointments for a patient", lambda patient_ref: {"patient_ref":patient_ref})
register_tool("get_referrals", "Read referrals for a patient", lambda patient_ref: {"patient_ref":patient_ref})

print(list(tool_registry))

['get_patient', 'get_plan', 'get_appointments', 'get_referrals']


## Agent roles

```text
                 Supervisor
                /    |     \
        Intake Agent  Policy Agent  Operations Agent
                \     |      /
                 Shared state
                       |
                    MCP tools
```

- **Intake Agent:** parses and classifies incoming requests.
- **Policy Agent:** retrieves operational policy.
- **Operations Agent:** uses appointments/referrals/eligibility tools.
- **Supervisor:** chooses the route and prevents agents from crossing their scope.

In [2]:
def supervisor(message):
    q=message.lower()
    if any(x in q for x in ["should i","dose","dosage","medication","interpret my symptoms"]):
        return {"route":"refuse_and_human_review","reason":"clinical advice request"}
    if "appointment" in q or "referral" in q:
        return {"route":"operations_agent"}
    return {"route":"policy_agent"}

for q in [
    "Show my next Cardiology appointment",
    "What does my plan cover?",
    "Should I take another tablet?"
]:
    print(q, "=>", supervisor(q))

Show my next Cardiology appointment => {'route': 'operations_agent'}
What does my plan cover? => {'route': 'policy_agent'}
Should I take another tablet? => {'route': 'refuse_and_human_review', 'reason': 'clinical advice request'}


### MCP boundary

The notebook models the **tool contract** that an MCP server would expose.

In a full implementation, replace the local registry with an MCP server whose tools map to the same deterministic functions. The agent layer should not directly access CSV files or database credentials.